In [ ]:
# STEP 1: Edit this list, then run this cell (Shift+Enter).
# One dictionary per league/season. Copy a line to add another; delete one to skip it.
# nickname = folder label; league_id = league ID in quotes; provider defaults to "yahoo"; year = season year.
# Yahoo league IDs can change each year. These entries are for 2025.
# Optional template: "original" (default), "pc", or "both" for two HTML reports.
# LEAGUES = [
#     {"nickname": "PHFFL_A", "league_id": "707737", "year": 2025, "template": "original"},
#     {"nickname": "CFFL_A", "league_id": "134317", "year": 2025, "template": "original"},
#     {"nickname": "CFFL_B", "league_id": "918145", "year": 2025, "template": "original"},
#     {"nickname": "Ferda", "league_id": "889216", "year": 2025, "template": "original"},
# ]
LEAGUES = [
    {"provider": "sleeper", "nickname": "The_Future_1pct", "league_id": "1385727391816503296", "year": 2026, "template": "both"},
    # {"nickname": "PHFFL_A", "league_id": "707737", "year": 2025, "template": "original"},
    # {"nickname": "PHFFL_A", "league_id": "1103430", "year": 2024, "template": "original"},
    # {"nickname": "CFFL_A", "league_id": "134317", "year": 2025, "template": "original"},
    # {"nickname": "CFFL_A", "league_id": "801641", "year": 2024, "template": "original"},
    # {"nickname": "CFFL_A", "league_id": "786024", "year": 2023, "template": "original"},
    # {"nickname": "CFFL_A", "league_id": "241250", "year": 2022, "template": "original"},
    # {"nickname": "CFFL_A", "league_id": "647681", "year": 2021, "template": "original"},
    # {"nickname": "CFFL_A", "league_id": "248625", "year": 2020, "template": "original"},
    # {"nickname": "CFFL_A", "league_id": "4170", "year": 2019, "template": "original"},
    # {"nickname": "CFFL_A", "league_id": "65579", "year": 2018, "template": "original"},
    # {"nickname": "CFFL_A", "league_id": "214515", "year": 2017, "template": "original"},
    # {"nickname": "CFFL_A", "league_id": "644498", "year": 2016, "template": "original"},
    # {"nickname": "CFFL_A", "league_id": "511114", "year": 2015, "template": "original"},
    # {"nickname": "CFFL_B", "league_id": "918145", "year": 2025, "template": "original"},
    # {"nickname": "CFFL_B", "league_id": "344186", "year": 2024, "template": "original"},
    # {"nickname": "CFFL_B", "league_id": "142728", "year": 2023, "template": "original"},
    # {"nickname": "CFFL_B", "league_id": "15111", "year": 2022, "template": "original"},
    # {"nickname": "CFFL_B", "league_id": "595463", "year": 2021, "template": "original"},
    # {"nickname": "CFFL_B", "league_id": "564724", "year": 2020, "template": "original"},
    # {"nickname": "CFFL_B", "league_id": "550706", "year": 2019, "template": "original"},
    # {"nickname": "CFFL_B", "league_id": "560098", "year": 2018, "template": "original"},
    # {"nickname": "CFFL_B", "league_id": "274817", "year": 2017, "template": "original"},
    # {"nickname": "CFFL_B", "league_id": "645417", "year": 2016, "template": "original"},
    # {"nickname": "CFFL_B", "league_id": "511141", "year": 2015, "template": "original"},
    # {"nickname": "Ferda", "league_id": "889216", "year": 2025, "template": "original"},
    # {"nickname": "Ferda", "league_id": "340963", "year": 2024, "template": "original"},
    # {"nickname": "Ferda", "league_id": "320083", "year": 2023, "template": "original"},
    {"nickname": "Ferda", "league_id": "691545", "year": 2022, "template": "original"},
    {"nickname": "Ferda", "league_id": "673625", "year": 2021, "template": "original"},
    {"nickname": "Ferda", "league_id": "307268", "year": 2020, "template": "original"},
    {"nickname": "Ferda", "league_id": "485349", "year": 2019, "template": "original"},
    {"nickname": "Ferda", "league_id": "1072496", "year": 2018, "template": "original"},
    {"nickname": "Ferda", "league_id": "202646", "year": 2017, "template": "original"},
    {"nickname": "Ferda", "league_id": "823954", "year": 2016, "template": "original"},
    # {"nickname": "Ferda", "league_id": "569704", "year": 2015, "template": "original"},
]

In [ ]:
# STEP 2: Run this cell and wait for Done! No edits needed.
from pathlib import Path
from yahoo_fantasy_data.config import Settings, storage_league_name
from yahoo_fantasy_data.yahoo import backfill_season
from report_code import ReportProcessor
from report_code.publish import resolve_week, report_templates

root = Path.cwd().resolve()
if not (root / "report_code" / "processor.py").is_file():
    raise RuntimeError("Open this notebook from the repository root, then run both cells.")
if "LEAGUES" not in globals() or not LEAGUES:
    raise RuntimeError("Run cell 1 first with at least one league in LEAGUES.")

# Public access only: no environment settings or login credentials are loaded.
settings = Settings(data_dir=root / "yahoo-fantasy-data" / "data", request_delay=2)
docs = root / "docs"
docs.mkdir(parents=True, exist_ok=True)

# Check template choices before downloading anything.
for league in LEAGUES:
    report_templates(league.get("template", "original"))
    if league.get("provider", "yahoo") not in ("yahoo", "sleeper"):
        raise ValueError("provider must be yahoo or sleeper")

for league in LEAGUES:
    nickname, league_id, year = league["nickname"], str(league["league_id"]), league["year"]
    folder = storage_league_name(nickname, league_id)
    archive = settings.data_dir / folder / str(year)
    week = resolve_week(league, archive, backfill=True, settings=settings)
    if week == 0:
        print(f"Skipping {nickname}, {year}: season has not started.")
        continue

    # 1. Download missing snapshots from week 1 through the latest completed week, capped at the regular-season end.
    print(f"Backfilling {nickname}, {year}, through week {week}...", flush=True)
    collect = backfill_season
    if league.get("provider") == "sleeper":
        from report_code.sleeper import backfill_season as collect
    statuses = collect(
        season=year, league_id=league_id, start_week=1, end_week=week,
        overwrite=False, settings=settings, league_nickname=nickname, refresh_latest=True,
    )
    if any(status.startswith("failed:") or status == "authentication_required"
           for weekly in statuses.values() for status in weekly.values()):
        raise RuntimeError(f"Incomplete backfill for {nickname}. Check statuses, then rerun cell 2.")

    # 2. Process the archived data and create the HTML report.
    report = ReportProcessor(archive, week)
    templates = report_templates(league.get("template", "original"))
    for template in templates:
        suffix = "_pc" if template == "pc" else ""
        report.write_html(docs / f"{folder}_{year}_week{week}{suffix}.html", template=template)

    # 3. Save the pandas DataFrames as gzip CSV files in docs/data/.
    data_folder = docs / "data" / folder / str(year)
    data_folder.mkdir(parents=True, exist_ok=True)
    report.silver_player.to_csv(data_folder / "silver_player.csv.gz", index=False, compression="gzip")
    report.silver_schedule.to_csv(data_folder / "silver_schedule.csv.gz", index=False, compression="gzip")
    print(f"Saved report and CSV.gz files for {nickname}, {year}.", flush=True)

print("Done! Reports and CSV.gz files are saved in docs/.")


## How to use this notebook

1. **One-time setup:** Select the project's `.venv` Python kernel. If dependencies are missing, run `.venv/bin/python -m pip install ./yahoo-fantasy-data -r report_code/requirements.txt` in a terminal from the repository root. For another environment, use its Python instead.
2. **Edit cell 1:** Each dictionary needs `nickname`, `league_id`, and `year`. Copy a line to add a league/season; delete a line to skip it. League IDs can change each year. Use a unique nickname for each league in the same year.
3. **Choose report wording:** Set `"template": "original"` for the standard report, `"template": "pc"` for only the work-appropriate version, or `"template": "both"` for both HTML versions. Omit `template` to use the original. The PC version changes report wording; league, manager, and player names remain as supplied.
4. **Run cell 1, then cell 2:** Press **Shift+Enter** in each cell, or choose **Run All**. Open this notebook from the repository root. Internet access is required; wait for `Done!`. Cell 2 calls `backfill_season()`, `ReportProcessor`, `write_html()`, and pandas `to_csv()` directly.
5. **View the results:** Open the HTML reports in `docs/`. For the download page, run `python3 -m http.server --directory docs` from the repository root and visit <http://localhost:8000>. This enables the “Download CSV (unzip)” links. Press **Ctrl+C** in the terminal to stop serving.

### Files saved — weekly reports and latest data

- **Original HTML:** `docs/NICKNAME_YEAR_weekN.html`.
- **PC HTML (when selected):** `docs/NICKNAME_YEAR_weekN_pc.html`.
- **Data:** `docs/data/NICKNAME/YEAR/silver_player.csv.gz` and `silver_schedule.csv.gz`. These are gzip-compressed CSVs written with `to_csv(index=False, compression="gzip")`. Pandas reads them directly with `pd.read_csv(path)`.
- **Download page:** `docs/index.html`.
- **Source archives:** `yahoo-fantasy-data/data/NICKNAME/YEAR/`. These retain historical snapshots for backfills and report calculations.

Each run saves HTML for the resolved week and preserves reports from earlier weeks. Rerunning the same week replaces that week’s HTML. CSV.gz files always overwrite the same paths for that league/season, keeping only the latest data export. No folder cleanup is needed. If you change from both report versions to just one, delete the unwanted HTML file manually. The index fetches its file list from GitHub when opened; no rebuild is needed. Local-only files will not appear in that list until pushed. Open local report files directly to preview them. No commit, push, or deployment is performed.

The default report week is `min(current_week - 1, playoff_start_week - 1)` using provider metadata and league settings. For example, current week 17 with playoffs starting in week 15 produces a week 14 report. The active week is excluded. Finished seasons include their final week, still capped at the regular-season cutoff. A run with no completed weeks is skipped. Leagues explicitly configured without playoffs use `end_week` as the cutoff. Missing metadata produces an error. Backfill downloads missing snapshots from week 1 through the selected week; existing source snapshots are reused. The current player export includes archived weeks through the report week, and schedules may include future matchups. “Latest” means one current export, not only one week's rows.

### Troubleshooting

- After editing the league list, rerun both cells. Different entries can use different years.
- This notebook uses public access only. It does not read `.env` or use login credentials. A league that requires authentication cannot be collected here.
- If Yahoo rate-limits a request or the connection fails, wait and rerun cell 2. Saved source archives are reused. A failed backfill stops before report processing; inspect `statuses` for details.
- If you see `ModuleNotFoundError`, install dependencies in the selected kernel's environment, restart the kernel, and run both cells again.

### Sleeper leagues

Add `"provider": "sleeper"` to a league entry; omitted providers use Yahoo. Sleeper needs no login. The Future 1% is configured above. See the root README for historical reserve/taxi and projection limitations.


In [ ]:
print(statuses)